In [1]:
import os
import platform
import re
import subprocess

class NetworkScanner:
    def __init__(self):
        self.os_type = platform.system().lower()
        self.arp_commands = {
            'windows': 'arp -a',
            'linux': 'arp -n',
            'darwin': 'arp -n'  # macOS
        }

    def get_arp_table(self):

        try:
            command = self.arp_commands.get(self.os_type, 'arp -a')
            result = subprocess.check_output(command, shell=True, stderr=subprocess.DEVNULL, universal_newlines=True)
            

            parsers = {
                'windows': self._parse_windows_arp,
                'linux': self._parse_unix_arp,
                'darwin': self._parse_unix_arp
            }
            
            parser = parsers.get(self.os_type, self._parse_unix_arp)
            return parser(result)
        
        except Exception as e:
            print(f"Error retrieving ARP table: {e}")
            return []

    def _parse_windows_arp(self, result):
        entries = []
        for line in result.split('\n')[3:]:  
            parts = line.split()
            if len(parts) >= 3 and re.match(r'\d+\.\d+\.\d+\.\d+', parts[0]):
                entries.append({
                    'IP_Address': parts[0], 
                    'MAC_Address': parts[1], 
                    'Type': parts[2]
                })
        return entries

    def _parse_unix_arp(self, result):
        entries = []
        for line in result.split('\n')[1:]:  
            parts = line.split()
            if len(parts) >= 4 and re.match(r'\d+\.\d+\.\d+\.\d+', parts[0]):
                entries.append({
                    'IP_Address': parts[0], 
                    'MAC_Address': parts[3], 
                    'Type': parts[2]
                })
        return entries

    def display_arp_table(self):
        arp_entries = self.get_arp_table()
        
        if not arp_entries:
            print("Unable to retrieve ARP table.")
            return
        
        print(f"{'IP Address':<20}{'MAC Address':<20}{'Type':<10}")
        print("-" * 50)
        
        for entry in arp_entries:
            print(f"{entry['IP_Address']:<20}{entry['MAC_Address']:<20}{entry['Type']:<10}")

def main():
    scanner = NetworkScanner()
    scanner.display_arp_table()

if __name__ == "__main__":
    main()

IP Address          MAC Address         Type      
--------------------------------------------------
192.168.209.8       12-22-af-a7-7b-03   dynamic   
192.168.209.28      14-ac-60-4b-0c-a5   dynamic   
192.168.209.171     ee-67-cd-24-4b-59   dynamic   
192.168.209.255     ff-ff-ff-ff-ff-ff   static    
224.0.0.22          01-00-5e-00-00-16   static    
224.0.0.251         01-00-5e-00-00-fb   static    
224.0.0.252         01-00-5e-00-00-fc   static    
255.255.255.255     ff-ff-ff-ff-ff-ff   static    
192.168.56.255      ff-ff-ff-ff-ff-ff   static    
224.0.0.22          01-00-5e-00-00-16   static    
224.0.0.251         01-00-5e-00-00-fb   static    
224.0.0.252         01-00-5e-00-00-fc   static    
239.255.255.250     01-00-5e-7f-ff-fa   static    
172.18.255.255      ff-ff-ff-ff-ff-ff   static    
224.0.0.22          01-00-5e-00-00-16   static    
224.0.0.251         01-00-5e-00-00-fb   static    


In [2]:
import platform
import subprocess
import re
import ipaddress

class ARPAnalyzer:
    def __init__(self):
        self.os_type = platform.system().lower()
        self.arp_commands = {
            'windows': 'arp -a',
            'linux': 'arp -n',
            'darwin': 'arp -n'
        }

    def get_arp_table(self):
        """Retrieve and parse ARP table across different platforms."""
        try:
            command = self.arp_commands.get(self.os_type, 'arp -a')
            result = subprocess.check_output(command, shell=True, stderr=subprocess.DEVNULL, text=True)
            
            return self._parse_arp_table(result)
        except Exception as e:
            print(f"ARP table retrieval error: {e}")
            return []

    def _parse_arp_table(self, result):
        """Parse ARP table based on operating system."""
        entries = []
        parser_func = (
            self._parse_windows_entries 
            if self.os_type == 'windows' 
            else self._parse_unix_entries
        )
        return parser_func(result)

    def _parse_windows_entries(self, result):
        """Parse Windows ARP table entries."""
        entries = []
        for line in result.split('\n')[3:]:
            parts = line.split()
            if len(parts) >= 3 and self._is_valid_ip(parts[0]):
                entries.append({
                    'IP Address': parts[0],
                    'MAC Address': parts[1],
                    'Type': parts[2]
                })
        return entries

    def _parse_unix_entries(self, result):
        """Parse Unix/Linux ARP table entries."""
        entries = []
        for line in result.split('\n')[1:]:
            parts = line.split()
            if len(parts) >= 4 and self._is_valid_ip(parts[0]):
                entries.append({
                    'IP Address': parts[0],
                    'MAC Address': parts[3],
                    'Type': parts[2]
                })
        return entries

    def _is_valid_ip(self, ip):
        """Validate IP address format."""
        try:
            ipaddress.ip_address(ip)
            return True
        except ValueError:
            return False

    def find_duplicate_macs(self, arp_table):
        """Detect devices with duplicate MAC addresses."""
        mac_map = {}
        for entry in arp_table:
            mac = entry['MAC Address']
            ip = entry['IP Address']
            mac_map.setdefault(mac, []).append(ip)
        
        return {mac: ips for mac, ips in mac_map.items() if len(ips) > 1}

def main():
    analyzer = ARPAnalyzer()
    arp_table = analyzer.get_arp_table()
    
    if not arp_table:
        print("No ARP entries found.")
        return

    print("Network ARP Table:")
    for entry in arp_table:
        print(f"IP: {entry['IP Address']} | MAC: {entry['MAC Address']} | Type: {entry['Type']}")

    duplicates = analyzer.find_duplicate_macs(arp_table)
    if duplicates:
        print("\nDuplicate MAC Addresses:")
        for mac, ips in duplicates.items():
            print(f"MAC {mac} found on IPs: {', '.join(ips)}")

if __name__ == "__main__":
    main()

Network ARP Table:
IP: 192.168.209.8 | MAC: 12-22-af-a7-7b-03 | Type: dynamic
IP: 192.168.209.28 | MAC: 14-ac-60-4b-0c-a5 | Type: dynamic
IP: 192.168.209.171 | MAC: ee-67-cd-24-4b-59 | Type: dynamic
IP: 192.168.209.255 | MAC: ff-ff-ff-ff-ff-ff | Type: static
IP: 224.0.0.22 | MAC: 01-00-5e-00-00-16 | Type: static
IP: 224.0.0.251 | MAC: 01-00-5e-00-00-fb | Type: static
IP: 224.0.0.252 | MAC: 01-00-5e-00-00-fc | Type: static
IP: 255.255.255.255 | MAC: ff-ff-ff-ff-ff-ff | Type: static
IP: 192.168.56.255 | MAC: ff-ff-ff-ff-ff-ff | Type: static
IP: 224.0.0.22 | MAC: 01-00-5e-00-00-16 | Type: static
IP: 224.0.0.251 | MAC: 01-00-5e-00-00-fb | Type: static
IP: 224.0.0.252 | MAC: 01-00-5e-00-00-fc | Type: static
IP: 239.255.255.250 | MAC: 01-00-5e-7f-ff-fa | Type: static
IP: 172.18.255.255 | MAC: ff-ff-ff-ff-ff-ff | Type: static
IP: 224.0.0.22 | MAC: 01-00-5e-00-00-16 | Type: static
IP: 224.0.0.251 | MAC: 01-00-5e-00-00-fb | Type: static

Duplicate MAC Addresses:
MAC ff-ff-ff-ff-ff-ff found on I

In [3]:
import paramiko
import re

def check_remote_mac_duplicates(target_ip, username, password):
    """
    Check for MAC address duplicates on a remote device via SSH.
    
    Args:
        target_ip (str): IP of target device
        username (str): SSH username
        password (str): SSH password
    
    Returns:
        dict: Duplicate MAC addresses with corresponding IPs
    """
    try:
        client = paramiko.SSHClient()
        client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        client.connect(target_ip, username=username, password=password)
        
        # Execute ARP table retrieval command
        stdin, stdout, stderr = client.exec_command('arp -a')
        arp_result = stdout.read().decode()
        client.close()
        
        # Parse ARP table and detect duplicates
        mac_map = {}
        for line in arp_result.split('\n'):
            match = re.search(r'(\d+\.\d+\.\d+\.\d+)\s+([0-9a-fA-F:]+)', line)
            if match:
                ip, mac = match.groups()
                mac_map.setdefault(mac, []).append(ip)
        
        # Return only duplicates
        return {mac: ips for mac, ips in mac_map.items() if len(ips) > 1}
    
    except Exception as e:
        print(f"Error: {e}")
        return {}

def main():
    target_ip = input("Enter remote device IP: ")
    username = input("Enter SSH username: ")
    password = input("Enter SSH password: ")
    
    duplicates = check_remote_mac_duplicates(target_ip, username, password)
    
    if duplicates:
        print("\nDuplicate MAC Addresses:")
        for mac, ips in duplicates.items():
            print(f"MAC {mac} found on IPs: {', '.join(ips)}")
    else:
        print("No MAC address duplicates found.")

if __name__ == "__main__":
    main()

Error: [Errno 11001] getaddrinfo failed
No MAC address duplicates found.
